# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Heart Disease (Cleveland)** dari UCI Machine Learning Repository.

Dataset ini berisi 303 sampel dengan 13 fitur klinis pasien untuk memprediksi keberadaan penyakit jantung.

- **Target:** 0 = tidak ada penyakit jantung, 1 = ada penyakit jantung
- **Task:** Klasifikasi Binary
- **Sumber:** UCI ML Repository (Cleveland database)

# **2. Import Library**

In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# **3. Memuat Dataset**

In [16]:
# Load dataset dari UCI Repository
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
    'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
]
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
df = pd.read_csv(url, header=None, names=columns, na_values='?')

# Convert target ke binary (0: no disease, 1: disease)
df['target'] = (df['target'] > 0).astype(int)

print(f"Shape dataset: {df.shape}")
df.head()

Shape dataset: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


In [ ]:
# Simpan dataset raw
df.to_csv('../heart_disease_raw/heart_disease_raw.csv', index=False)
print("Dataset raw berhasil disimpan.")

OSError: Cannot save file into a non-existent directory: '../namadataset_raw'

# **4. Exploratory Data Analysis (EDA)**

In [ ]:
# Info dataset
print("=== INFO DATASET ===")
print(df.info())
print("\n=== STATISTIK DESKRIPTIF ===")
df.describe()

In [ ]:
# Cek missing values
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print(f"\n=== DUPLIKASI ===")
print(f"Jumlah data duplikat: {df.duplicated().sum()}")

print(f"\n=== DISTRIBUSI TARGET ===")
print(df['target'].value_counts())
print(f"\nRasio: {df['target'].value_counts(normalize=True).to_dict()}")

In [ ]:
# Korelasi dengan target
print("=== KORELASI DENGAN TARGET ===")
corr = df.corr()['target'].sort_values(ascending=False)
print(corr)

In [ ]:
# Deteksi outlier menggunakan IQR
print("=== DETEKSI OUTLIER (IQR) ===")
numeric_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outlier(s)")

# **5. Data Preprocessing**

Tahapan preprocessing:
1. Menangani missing values (median imputation)
2. Menghapus data duplikat
3. Menangani outlier dengan IQR clipping
4. Standarisasi fitur numerik kontinu
5. Split data menjadi train dan test set (80:20)

In [ ]:
# 1. Handle missing values
print("=== HANDLING MISSING VALUES ===")
for col in df.columns:
    if df[col].isnull().sum() > 0:
        print(f"{col}: {df[col].isnull().sum()} missing -> filled with median")
        df[col] = df[col].fillna(df[col].median())

print(f"Missing values setelah handling: {df.isnull().sum().sum()}")

In [ ]:
# 2. Hapus duplikat
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplikat dihapus: {before - len(df)} baris")

In [ ]:
# 3. Handling outlier dengan IQR clipping
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

print("Outlier berhasil di-clip.")

In [ ]:
# 4. Split fitur dan target
X = df.drop(columns=['target'])
y = df['target']

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Standarisasi fitur numerik
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_test shape: {X_test_scaled.shape}")
print(f"\nDistribusi target train:\n{y_train.value_counts()}")

In [ ]:
# Simpan hasil preprocessing
train_df = X_train_scaled.copy().reset_index(drop=True)
train_df['target'] = y_train.values
test_df = X_test_scaled.copy().reset_index(drop=True)
test_df['target'] = y_test.values

train_df.to_csv('heart_disease_preprocessing/heart_disease_train.csv', index=False)
test_df.to_csv('heart_disease_preprocessing/heart_disease_test.csv', index=False)
print("Dataset preprocessing berhasil disimpan.")
print(f"Train: {train_df.shape}, Test: {test_df.shape}")